# 🧩 이미지 슬라이드 → 편집 가능한 PPTX (slide-unflatten)

NotebookLM·Gemini·Canva·Gamma 등에서 받은 **통이미지 슬라이드(PPTX/PDF/PNG)** 를 글자와 그림을 고칠 수 있는 PPTX로 바꿉니다.

## 🙋 처음 한 번만
1. **파일 → 드라이브에 사본 저장** 후 사본에서 작업하세요.
2. **런타임 → 런타임 유형 변경 → T4 GPU → 저장** (없어도 되지만 느립니다).
3. *(선택, 강력 추천)* Claude API 키: [console.anthropic.com](https://console.anthropic.com) → Billing 충전 → API Keys → 키 복사 → 왼쪽 **🔑 보안 비밀**에 이름 `ANTHROPIC_API_KEY`, 값 = 키, **노트북 액세스** 켜기.
   키가 있으면 오타 교정·문단 묶기·그림 개체 분리까지 됩니다. 슬라이드 1장당 약 60원(Opus) / 30원(Sonnet).

## ▶️ 매번
**런타임 → 모두 실행** → 4번 셀에서 파일 선택 → 끝나면 `*_editable.pptx`와 리포트가 자동 다운로드됩니다.
웹 화면이 편하면 맨 아래 **웹 UI** 셀을 실행하세요.

> 🔒 파일은 내 Colab 작업 공간에서 처리됩니다. Claude를 쓰면 슬라이드 이미지가 Anthropic으로 전송됩니다.

## 1. 설치 (약 2분)

In [ ]:
#@title 설치 { display-mode: "form" }
REPO_URL = "https://github.com/Leewanglyul/slide-unflatten.git"  #@param {type:"string"}
import subprocess, sys, importlib
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "-q", "install", *a], check=False)
pip(f"slide-unflatten[claude,ocr,segment,ui] @ git+{REPO_URL}")
pip("--no-deps", "simple-lama-inpainting")          # 옛 Pillow 핀 때문에 --no-deps
subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "fonts-noto-cjk"], capture_output=True)
try:
    importlib.import_module("unflatten")
except ImportError:
    print("⚠️ git 설치 실패. 패키지 zip을 올려 주세요 (slide-unflatten 폴더를 zip으로).")
    from google.colab import files
    up = files.upload()
    for n in up:
        subprocess.run(["unzip", "-oq", n]); pip("-e", n[:-4] + "[claude,ocr,segment,ui]")
from unflatten.cli import doctor
doctor()

## 2. Claude API 키 확인
`API key: ✅`가 나오면 정상입니다. 없어도 규칙 기반으로 계속 진행됩니다.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = (userdata.get("ANTHROPIC_API_KEY") or "").strip().strip('"').strip("'")
except Exception as e:
    print("보안 비밀을 읽지 못했습니다:", e)
from unflatten.config import Config
from unflatten.claude_client import ClaudeService
USE_CLAUDE = False
if os.environ.get("ANTHROPIC_API_KEY"):
    ok, msg = ClaudeService(Config()).validate()
    USE_CLAUDE = ok
    print("API key:", "✅" if ok else "❌", msg)
else:
    print("API 키 없음 → 규칙 기반으로 진행합니다 (보안 비밀 이름 ANTHROPIC_API_KEY · 노트북 액세스 확인)")

## 3. 설정

아래 폼에서 값을 고른 뒤 셀을 실행하세요. 잘 모르겠으면 **기본값 그대로** 두면 됩니다.

| 설정 | 기본값 | 무엇을 하나요 | 이럴 때 바꾸세요 |
|---|---|---|---|
| **MODE** | `full` | `text`: 글자만 편집 가능하게 만들고 그림은 배경에 그대로 둡니다. 가장 빠르고 저렴합니다.<br>`objects`: 글자 + 그림을 덩어리(삽화 하나, 사진 한 장) 단위로 떼어 냅니다.<br>`full`: 그림 속 개별 대상(사람, 물건)까지 따로 떼어 냅니다. 앞뒤로 겹친 부분은 AI가 채워 넣어, 앞의 것을 옮겨도 구멍이 보이지 않습니다. | 글자만 고치면 되면 `text`. 그림이 너무 잘게 쪼개지면 `objects`. |
| **LANGUAGE** | `ko` | 슬라이드에 쓰인 언어입니다. 글자 인식과 기본 글꼴을 정합니다. | 영어·일본어·중국어 자료일 때 |
| **MODEL** | `claude-opus-5-5` | 오타 교정·문단 묶기·그림 위치를 판단할 Claude 모델입니다. API 키가 있을 때만 씁니다.<br>Opus 5.5: 가장 정확, 장당 약 50~80원 · Sonnet 5.5: 비용 절반 | 비용을 줄이고 싶으면 Sonnet |
| **SLIDES** | (비움) | 변환할 슬라이드 번호입니다. 예: `1-3,7`. 비우면 전체를 변환합니다. 지정하지 않은 장은 원본 그대로 남습니다. | 몇 장만 시험해 볼 때 |
| **KEEP_HIDDEN_ORIGINAL** | 켬 | 원본 이미지를 각 슬라이드 맨 아래에 **숨겨서** 넣어 둡니다. PowerPoint **홈 → 정렬 → 선택 창**에서 켜면 원본과 겹쳐 비교할 수 있습니다. | 파일 용량을 줄이려면 끔 |
| **GROUP_SHAPES** | 켬 | 떼어 낸 그림들은 `그림` 그룹, 글자 상자들은 `텍스트` 그룹으로 묶어 선택 창을 정리합니다. 그룹 안의 개체는 **한 번 더 클릭**하면 하나씩 선택됩니다. | 개체를 하나씩 바로 선택하고 싶으면 끔 |
| **THEME_FONTS** | 켬 | 글꼴을 PowerPoint **테마 글꼴**로 지정합니다. 나중에 **디자인 → 변형 → 글꼴**에서 한 번에 바꿀 수 있습니다. (이미 편집 가능한 슬라이드가 섞인 PPTX는 원래 디자인을 지키려고 자동으로 끕니다.) | 글꼴 이름을 직접 박아 넣고 싶으면 끔 |
| **REMOVE_WATERMARK** | 켬 | 슬라이드 구석의 생성 도구 표시(로고 글씨)를 지웁니다. Claude가 있을 때만 작동합니다. | 표시를 남기려면 끔 |
| **UPSCALE** | `1` | 배경·그림 조각 이미지를 몇 배로 키워 넣을지입니다. `2`는 큰 화면에서 덜 뭉개 보이지만, 단순 확대라 **선명도가 실제로 좋아지지는 않고** 용량만 약 4배가 됩니다. | 대형 화면 발표용일 때만 `2` |
| **EDGE_EXPAND** | `4` | 그림 조각의 **가장자리를 몇 px 넓혀** 외곽선·스티커의 흰 테두리까지 포함할지입니다. 배경색(카드 색)은 넓혀도 따라오지 않습니다. | 흰 테두리가 잘려 보이면 ↑ (6~8) / 조각 둘레에 배경색 띠가 붙으면 ↓ (2) / `0`이면 끔 |
| **EDGE_FEATHER** | `1.0` | 그림 조각 테두리를 몇 px **부드럽게** 할지입니다. `0`은 칼로 자른 듯 계단이 보이고, 너무 크면 흐릿해집니다. | 테두리가 계단처럼 거칠면 ↑ (1.5~2) / 흐릿하면 ↓ |
| **UPLOAD_FONTS** | 켬 | 실행하면 글꼴 파일(.ttf/.otf) 업로드 창이 뜹니다. **보통 굵기 + 굵은 것** 두 개를 함께 올리면 본문·제목에 자동 배정되고, 글자 크기도 그 글꼴 폭에 맞춰집니다. 취소하면 언어 기본 글꼴을 씁니다. | 기본 글꼴로 충분하면 끔 |

> 💡 **EDGE_EXPAND / EDGE_FEATHER만 바꿀 때는 처음부터 다시 변환하지 마세요.** 값을 바꿔 이 셀을 실행한 뒤 **5번 재조립 셀**을 실행하면, 인식 결과를 재사용해서 그림 자르기만 다시 합니다. Claude 비용이 들지 않습니다.


In [ ]:
#@title ⚙️ 변환 설정 { display-mode: "form" }
#@markdown **변환 범위** — 표의 설명을 참고하세요. 모르면 기본값 그대로.
MODE = "full"  #@param ["text", "objects", "full"] {type:"string"}
#@markdown `text` 글자만 · `objects` 글자 + 그림 덩어리 · `full` 그림 속 사람·물건까지 따로
LANGUAGE = "ko"  #@param ["ko", "en", "ja", "zh"]
#@markdown 슬라이드에 쓰인 언어 (글자 인식·기본 글꼴)
MODEL = "claude-opus-5-5"  #@param ["claude-opus-5-5", "claude-sonnet-5-5"]
#@markdown Opus 가장 정확(장당 약 50~80원) · Sonnet 비용 절반 — API 키가 있을 때만 사용
SLIDES = ""  #@param {type:"string"}
#@markdown 변환할 장 (예: `1-3,7`). 비우면 전체
#@markdown ---
#@markdown **결과 파일 모양**
KEEP_HIDDEN_ORIGINAL = True  #@param {type:"boolean"}
#@markdown 원본 이미지를 숨김 층으로 보관(비교용). 끄면 용량 감소
GROUP_SHAPES = True  #@param {type:"boolean"}
#@markdown 그림끼리·글자끼리 그룹으로 묶기. 끄면 낱개로 바로 선택
THEME_FONTS = True  #@param {type:"boolean"}
#@markdown 테마 글꼴로 지정 → 디자인 탭에서 한 번에 글꼴 변경 가능
REMOVE_WATERMARK = True  #@param {type:"boolean"}
#@markdown 구석의 생성 도구 표시 지우기 (Claude 필요)
UPSCALE = 1  #@param [1, 2] {type:"raw"}
#@markdown 이미지 확대 배율. 2는 용량 약 4배, 선명도는 그대로
#@markdown ---
#@markdown **그림 조각 가장자리** — 바꾼 뒤 5번 재조립 셀만 실행하면 비용 없이 다시 자릅니다
EDGE_EXPAND = 4  #@param {type:"slider", min:0, max:12, step:1}
#@markdown 외곽선·흰 스티커 테두리를 몇 px까지 포함할지. 테두리가 잘리면 ↑, 배경색 띠가 붙으면 ↓
EDGE_FEATHER = 1.0  #@param {type:"slider", min:0, max:3, step:0.5}
#@markdown 테두리를 몇 px 부드럽게 할지. 계단처럼 거칠면 ↑, 흐릿하면 ↓, 0이면 딱딱한 경계
#@markdown ---
UPLOAD_FONTS = True  #@param {type:"boolean"}
#@markdown 글꼴 파일(.ttf/.otf) 업로드 창 띄우기 — 보통 + 굵게 두 개를 함께 올리면 본문·제목에 자동 배정 (취소하면 기본 글꼴)

from unflatten.config import Config
from unflatten.fonts import assign_font_files, font_family
_prev_fonts = (cfg.font_regular_file, cfg.font_bold_file) if "cfg" in globals() else (None, None)
cfg = Config(mode=MODE, language=LANGUAGE, model=MODEL, use_claude=USE_CLAUDE, slide_range=SLIDES or None,
             keep_hidden_original=KEEP_HIDDEN_ORIGINAL, group_shapes=GROUP_SHAPES, use_theme_fonts=THEME_FONTS,
             remove_watermark=REMOVE_WATERMARK, upscale=int(UPSCALE), edge_expand=int(EDGE_EXPAND),
             edge_feather=float(EDGE_FEATHER), out_dir="/content/unflatten_out")
cfg.font_regular_file, cfg.font_bold_file = _prev_fonts
if UPLOAD_FONTS:
    from google.colab import files
    print("글꼴 파일을 선택하세요 (건너뛰려면 취소 — 이전에 올린 글꼴이 있으면 그대로 씁니다)")
    try:
        up = files.upload()
    except Exception:
        up = {}
    if up:
        reg, bold = assign_font_files(list(up))
        cfg.font_regular_file, cfg.font_bold_file = reg, bold
if cfg.font_regular_file:
    print("본문 글꼴:", font_family(cfg.font_regular_file), "| 제목 글꼴:",
          font_family(cfg.font_bold_file) if cfg.font_bold_file else "(본문 + 굵게)")
print("설정 완료:", cfg.mode, cfg.language, cfg.model if cfg.use_claude else "Claude 없음",
      f"| 가장자리 넓힘 {cfg.edge_expand}px · 부드럽게 {cfg.edge_feather}px")


## 4. 파일 올리고 변환하기 (PPTX · PDF · PNG/JPG, 여러 개 가능)

In [ ]:
from google.colab import files
from unflatten.pipeline import convert, estimate, bundle_zip
from unflatten.cli import print_summary
uploaded = files.upload()
for name in uploaded:
    print("=====", name, "|", estimate(name, cfg)["line"])
results = []
for name in uploaded:
    try:
        res = convert(name, cfg)
        print_summary(res)
        results.append(res)
    except Exception as e:
        print("❌", name, type(e).__name__, e)
for res in results:
    files.download(res.out_path)
    if res.report_path: files.download(res.report_path)
    if res.font_files: files.download(bundle_zip(res))

## 5. 결과 확인 · 수정 · 재조립
- `*_report.html`: 슬라이드별 상태와 원본/결과 비교. 경고가 있는 장을 먼저 보세요.
- **글꼴, 그룹, 숨김 원본, 그림 가장자리(EDGE_EXPAND·EDGE_FEATHER)** 를 바꾸려면 3번 셀에서 값을 바꿔 실행한 뒤 아래 셀을 실행하세요.
  인식 결과(OCR·Claude)를 재사용하므로 **Claude 비용이 다시 들지 않습니다.** 가장자리를 바꾼 경우 그림 자르기만 다시 하므로 장당 몇 초가 걸립니다.


In [ ]:
from unflatten.pipeline import rebuild
from unflatten.cli import print_summary
from google.colab import files
for res in results:
    r2 = rebuild(res.manifest_path, {"font_regular_file": cfg.font_regular_file, "font_bold_file": cfg.font_bold_file,
                                     "group_shapes": cfg.group_shapes, "use_theme_fonts": cfg.use_theme_fonts,
                                     "keep_hidden_original": cfg.keep_hidden_original, "upscale": cfg.upscale,
                                     "edge_expand": cfg.edge_expand, "edge_feather": cfg.edge_feather})
    print_summary(r2); files.download(r2.out_path)


## (선택) 웹 UI
업로드 → 옵션 → 예상 비용 → 변환 → 미리보기 → 다운로드가 한 화면에 있습니다. 텍스트 수정·재조립 탭도 있습니다.

In [ ]:
from unflatten.app import launch
launch(share=True, inline=True, ui_lang="ko")